# Beyond - Multimodal Language Models

This notebook slices MNIST digits into patches, splices them into a from-scratch StoryLM-1M-class transformer's token stream, and trains it with the same masked next-token loss as Module 13. It then generates captions autoregressively, parses their digits for test accuracy, and compares the result with your Module 03 MLP. MNIST downloads automatically on first use (~12MB).

1. Read the lesson page (`docs/beyond/multimodal.md`).
2. Open this notebook with `./notebook.sh multimodal`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision import datasets

from g2c.multimodal import (
    IMG_ID, MultimodalLM, build_caption_batch, decode_ids, patchify,
)
from g2c.multimodal.vocab import VOCAB_SIZE
from g2c.transformer import TransformerLM

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

mnist_train = datasets.MNIST(root="data", train=True, download=True)
mnist_test = datasets.MNIST(root="data", train=False, download=True)
train_images, train_labels = mnist_train.data, mnist_train.targets
test_images, test_labels = mnist_test.data, mnist_test.targets
print(f"train {tuple(train_images.shape)}, test {tuple(test_images.shape)}")

## Exercise 1 — Patchify and look

Slice digits at three patch sizes and note the sequence-length cost of each choice — this is the "an image costs N tokens" line item from every VLM model card, held in your hand.

In [ ]:
image = train_images[0].float() / 255.0
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(image, cmap="gray")
axes[0].set_title(f"digit {int(train_labels[0])}")
for ax, p in zip(axes[1:], (14, 7, 4)):
    patches = patchify(image[None], p)[0]
    n = patches.shape[0]
    grid = int(n ** 0.5)
    tiled = patches.reshape(grid, grid, p, p).permute(0, 2, 1, 3)
    tiled = tiled.reshape(grid * p, grid * p)
    ax.imshow(tiled, cmap="gray")
    ax.set_title(f"p={p}: {n} tokens/image")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
"Question: At patch size 4 an image costs 49 tokens; at 14 it costs 4. Name one thing the model plausibly LOSES at the cheap setting and one cost that grows at the expensive one — and connect this to why production VLMs meter high-resolution images at hundreds of tokens."
"Answer: "

## Exercise 2 — Train the caption model

Train a StoryLM-1M-class backbone from scratch on `"<img>×16 This is a 7 . <end>"`, with masked loss on caption targets only. Here “StoryLM-1M-class” means its four-layer, `D=128`, four-head, `hidden=512` architecture; the tiny 17-token caption vocabulary makes the exact parameter count somewhat smaller. After training, generate held-out captions autoregressively rather than reading a teacher-forced digit slot.

In [ ]:
from g2c.sft import masked_cross_entropy
from g2c.training import AdamW, clip_grad_norm_, cosine_with_warmup

model_cfg = dict(
    vocab_size=VOCAB_SIZE, embedding_dim=128, num_layers=4,
    num_heads=4, max_seq_len=64, hidden_dim=512,
)
train_cfg = dict(
    steps=1_000, batch_size=64, max_lr=3e-4, min_lr=3e-5,
    warmup_steps=100, patch_size=7, log_every=100, seed=21,
)


def train_caption_model(*, image_transform=None, seed=21):
    torch.manual_seed(1)
    model = MultimodalLM(
        TransformerLM(**model_cfg), train_cfg["patch_size"]
    ).to(device)
    opt = AdamW(model.parameters(), train_cfg["max_lr"], weight_decay=0.05)
    gen = torch.Generator().manual_seed(seed)
    history = {"step": [], "train_loss": [], "val_loss": []}

    val_source = test_images[:512]
    if image_transform is not None:
        val_source = image_transform(val_source)
    vx, vimgs, vy, vmask = build_caption_batch(
        val_source, test_labels[:512], patch_size=train_cfg["patch_size"]
    )

    for step in range(1, train_cfg["steps"] + 1):
        sel = torch.randint(
            0, train_images.shape[0], (train_cfg["batch_size"],),
            generator=gen,
        )
        image_source = train_images[sel]
        if image_transform is not None:
            image_source = image_transform(image_source)
        x, imgs, y, mask = build_caption_batch(
            image_source, train_labels[sel],
            patch_size=train_cfg["patch_size"],
        )
        logits = model(x.to(device), imgs.to(device))
        loss = masked_cross_entropy(logits, y.to(device), mask.to(device))
        opt.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        opt.lr = cosine_with_warmup(
            step - 1, warmup_steps=train_cfg["warmup_steps"],
            max_steps=train_cfg["steps"], max_lr=train_cfg["max_lr"],
            min_lr=train_cfg["min_lr"],
        )
        opt.step()

        if step % train_cfg["log_every"] == 0:
            with torch.no_grad():
                val_loss = masked_cross_entropy(
                    model(vx.to(device), vimgs.to(device)),
                    vy.to(device), vmask.to(device),
                ).item()
            history["step"].append(step)
            history["train_loss"].append(float(loss.detach()))
            history["val_loss"].append(val_loss)
            print(f"step {step:>4}  train {loss.item():.3f}  val {val_loss:.3f}")
    return model, history


mm, history = train_caption_model()
print(f"trainable parameters: {sum(p.numel() for p in mm.parameters()):,}")
plt.plot(history["step"], history["train_loss"], label="train")
plt.plot(history["step"], history["val_loss"], label="validation")
plt.xlabel("step"); plt.ylabel("masked caption loss"); plt.legend()
plt.show()

## Exercise 3 — Score it as a classifier

Generate captions for the MNIST test set, parse the first digit token each caption emits, and compute accuracy. Put that measured number next to the result you recorded in Module 03; do not substitute a hard-coded reference. The comparison asks what the captioning route costs at this scale, not whether transformers are generally better or worse image classifiers.

In [ ]:
@torch.no_grad()
def generate_caption_ids(model, images, *, patch_size=7, max_new_tokens=6):
    if images.dtype == torch.uint8:
        images = images.float() / 255.0
    else:
        images = images.float()
    batch, height, width = images.shape
    n_patches = (height // patch_size) * (width // patch_size)
    token_ids = torch.full(
        (batch, n_patches), IMG_ID, dtype=torch.long, device=device
    )
    images = images.to(device)
    generated = []
    for _ in range(max_new_tokens):
        next_ids = model(token_ids, images)[:, -1].argmax(dim=-1)
        generated.append(next_ids.cpu())
        token_ids = torch.cat([token_ids, next_ids[:, None]], dim=1)
    return torch.stack(generated, dim=1)


def generated_digit_accuracy(
    model, images, labels, *, patch_size=7, chunk=512, image_transform=None
):
    correct = 0
    for i in range(0, len(labels), chunk):
        image_chunk = images[i : i + chunk]
        if image_transform is not None:
            image_chunk = image_transform(image_chunk)
        generated = generate_caption_ids(
            model, image_chunk, patch_size=patch_size
        )
        for row, label in zip(generated.tolist(), labels[i : i + chunk].tolist()):
            predicted = next((token - 7 for token in row if 7 <= token <= 16), -1)
            correct += int(predicted == label)
    return correct / len(labels)


sample_generated = generate_caption_ids(mm, test_images[:12])
for label, token_row in zip(test_labels[:12].tolist(), sample_generated.tolist()):
    print(f"true {label}: {decode_ids(token_row)}")

acc = generated_digit_accuracy(mm, test_images, test_labels)
print(f"generated-caption digit accuracy: {acc:.1%}")
print("Compare this measured value with the test accuracy you recorded in Module 03.")

In [ ]:
"Question: Report your generated-caption digit accuracy and the MNIST test accuracy you recorded in Module 03. The models see the same underlying pixels but optimize different objectives. What extra sequence-and-language behavior must the caption model learn, and what conclusion would be too broad to draw from this one comparison?"
"Answer: "

## Exercise 4 — Shuffle the patches

Retrain with patches in one fixed random order and compare generated-caption accuracy. A consistent permutation preserves slot identity but changes the spatial prior; report the measured effect without assuming it must be small.

In [ ]:
perm = torch.randperm(16, generator=torch.Generator().manual_seed(7))


def shuffle_patches(images):
    p = train_cfg["patch_size"]
    patches = patchify(images, p)[:, perm]
    grid = 28 // p
    return (
        patches.reshape(-1, grid, grid, p, p)
        .permute(0, 1, 3, 2, 4)
        .reshape(-1, 28, 28)
    )


mm_shuffled, shuffled_history = train_caption_model(
    image_transform=shuffle_patches
)
acc_shuffled = generated_digit_accuracy(
    mm_shuffled, test_images, test_labels,
    image_transform=shuffle_patches,
)
print(f"shuffled-patch accuracy: {acc_shuffled:.1%}  (ordered: {acc:.1%})")

In [ ]:
"Question: Compare the shuffled-patch generated-caption accuracy against the ordered result. What did the fixed permutation preserve, what spatial prior did it alter, and did the measured effect match your expectation?"
"Answer: "

## Exercise 5 — Bridge the toy to production

Our direct `Linear(49, D)` frontend isolates how visual vectors enter a language model, but it skips most of a production vision system. Identify exactly what transfers and what larger systems add before treating this notebook as a model-card decoder.

In [ ]:
"Question: Which interface idea from this MNIST model survives in a production VLM? Name at least three jobs typically handled by a vision tower, projector/resampler, resolution pipeline, or large-scale training. Why does the phrase 'native multimodal' not tell you whether a model uses a vision encoder?"
"Answer: "

## Exercise 6 — Two images, one sequence (optional)

Build a sequence containing two blocks of image placeholders and verify that both images splice cleanly into one transformer context. This is an interface smoke test, not evidence that the single-image-trained model learned relational binding. A genuine two-image captioning task and evaluation are left as an extension.

In [ ]:
from g2c.multimodal import caption_ids

n_patches = 16
ids = [IMG_ID] * n_patches + [IMG_ID] * n_patches + caption_ids(int(train_labels[1]))
x2 = torch.tensor([ids], dtype=torch.long)
imgs2 = torch.stack([train_images[0], train_images[1]]).float()[None] / 255.0
with torch.no_grad():
    logits2 = mm(x2.to(device), imgs2.to(device))
print(f"two-image splice smoke test: logits {tuple(logits2.shape)}")
print("This checks the interface only; the model was not trained for two-image binding.")
print(decode_ids(ids[-6:]))

When complete, ask a coding agent to grade your notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.